# Phase 3: Embedding & FAISS Indexing

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Hardware**: ⚡ Google Colab T4 GPU (REQUIRED)  
**Model**: `intfloat/multilingual-e5-large` (560M params · dim=1024)

**Parallelism**: Run this notebook on 3 separate Colab accounts simultaneously.  
Change `TARGET_STRATEGY` below to match the account assignment:

| Account | TARGET_STRATEGY |
|---------|-----------------|
| Account 1 | `fixed_size` |
| Account 2 | `sentence_aware` |
| Account 3 | `article_level` |

All cells are **idempotent** — safe to re-run after a Colab timeout.  
On resume, the checkpoint is detected automatically and encoding continues from where it stopped.

## Cell 0 — Environment Setup & GPU Verification

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

# ── Strategy selector (change this per Colab account) ──────────────────────
TARGET_STRATEGY = 'fixed_size'   # options: 'fixed_size' | 'sentence_aware' | 'article_level'

# ── Environment detection ───────────────────────────────────────────────────
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

# ── Environment Setup & Installations ───────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    GIT_DIR = Path('/content/rag-vn-finance')
    REPO_ROOT = GIT_DIR
    DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/rag-vn-finance/implementation')

    # Chỉ clone và install nếu thư mục chưa tồn tại để tránh phá vỡ bộ nhớ đệm
    if not GIT_DIR.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(GIT_DIR)])

        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')

        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp thư viện lõi (Numpy/Torch)...")
        os.kill(os.getpid(), 9) # Tự động ngắt tiến trình để ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Đang cập nhật code mới nhất từ Github...")
        subprocess.run(['git', '-C', str(GIT_DIR), 'pull'], capture_output=True)
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
    DRIVE_DATA_ROOT = REPO_ROOT

# ── Safe to import ML libraries now ─────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

src_path = REPO_ROOT / 'src'
if not src_path.exists():
    raise FileNotFoundError(f"LỖI: Không tìm thấy thư mục src tại {src_path}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Target strategy: {TARGET_STRATEGY}")
print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local'} | Device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mã nguồn đã tồn tại. Đang cập nhật code mới nhất từ Github...
Target strategy: fixed_size
Runtime: Google Colab | Device: cuda


## Cell 1 — Load Config & Resolve Paths

In [2]:
import json
import pandas as pd
from src.utils import load_config, resolve_path

config = load_config(REPO_ROOT / 'configs' / 'config.yaml')
emb_cfg = config['embedding']

# ── Resolve paths (SỬ DỤNG DRIVE_DATA_ROOT CHO DATA) ──────────────────────
chunks_base = resolve_path(config['chunking'], 'output_dir')
if not os.path.isabs(chunks_base):
    chunks_base = str(DRIVE_DATA_ROOT / chunks_base)

emb_base = resolve_path(emb_cfg, 'output_dir')
if not os.path.isabs(emb_base):
    emb_base = str(DRIVE_DATA_ROOT / emb_base)

idx_base = resolve_path(config['indexing'], 'output_dir')
if not os.path.isabs(idx_base):
    idx_base = str(DRIVE_DATA_ROOT / idx_base)

# Per-strategy paths
CHUNKS_PATH     = os.path.join(chunks_base, TARGET_STRATEGY, 'chunks.parquet')
EMB_DIR         = os.path.join(emb_base,    TARGET_STRATEGY)
EMB_NPY_PATH    = os.path.join(EMB_DIR,     'embeddings.npy')
CHECKPOINT_PATH = os.path.join(EMB_DIR,     'checkpoint.json')
IDX_DIR         = os.path.join(idx_base,    TARGET_STRATEGY)
FAISS_PATH      = os.path.join(IDX_DIR,     'index.faiss')

# Tạo folder nếu chưa có trên Drive
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(IDX_DIR, exist_ok=True)

print(f"Strategy         : {TARGET_STRATEGY}")
print(f"Chunks input     : {CHUNKS_PATH}")
print(f"Embeddings dir   : {EMB_DIR}")
print(f"Embeddings .npy  : {EMB_NPY_PATH}")
print(f"Checkpoint       : {CHECKPOINT_PATH}")
print(f"FAISS index      : {FAISS_PATH}")

assert os.path.exists(CHUNKS_PATH), f"❌ Chunks file not found: {CHUNKS_PATH}"
print("\n✅ Paths resolved. Cell 1 complete.")

Strategy         : fixed_size
Chunks input     : /content/drive/MyDrive/rag-vn-finance/data/chunks/fixed_size/chunks.parquet
Embeddings dir   : /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size
Embeddings .npy  : /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy
Checkpoint       : /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/checkpoint.json
FAISS index      : /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/index.faiss

✅ Paths resolved. Cell 1 complete.


## Cell 2 — Load Chunk Dataset

In [3]:
df_chunks = pd.read_parquet(CHUNKS_PATH)
print(f"Chunks loaded: {len(df_chunks):,} rows")
print(f"Columns      : {df_chunks.columns.tolist()}")
print(f"Sample chunk_id: {df_chunks['chunk_id'].iloc[0]}")

# Verify required columns
required = ['chunk_id', 'doc_id', 'text', 'strategy',
            'source', 'category', 'year', 'title', 'url',
            'tickers', 'is_historical', 'numerical_density', 'entities']
missing = [c for c in required if c not in df_chunks.columns]
if missing:
    raise ValueError(f"Missing columns in chunks: {missing}")

print(f"\n✅ Schema verified. {len(df_chunks):,} chunks ready.")
df_chunks.head(2)

Chunks loaded: 45,764 rows
Columns      : ['chunk_id', 'doc_id', 'text', 'chunk_index', 'total_chunks', 'strategy', 'source', 'category', 'time', 'year', 'title', 'url', 'tickers', 'is_historical', 'numerical_density', 'entities']
Sample chunk_id: b2eda821f172bbee_c0000

✅ Schema verified. 45,764 chunks ready.


,chunk_id,doc_id,text,chunk_index,total_chunks,strategy,source,category,time,year,title,url,tickers,is_historical,numerical_density,entities
0,b2eda821f172bbee_c0000,b2eda821f172bbee,"Title: Hàng về tranh bán, thị trường chìm tron...",0,4,fixed_size,vneconomy.vn,Chứng khoán,2023-03-03T00:00:00,2023,"Hàng về tranh bán, thị trường chìm trong sắc đ...",https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,[],False,0.070348,"1/3,cuối,2,hôm,Khoảng,Kết phiên,15,87 điểm,T,s..."
1,b2eda821f172bbee_c0001,b2eda821f172bbee,"Title: Hàng về tranh bán, thị trường chìm tron...",1,4,fixed_size,vneconomy.vn,Chứng khoán,2023-03-03T00:00:00,2023,"Hàng về tranh bán, thị trường chìm trong sắc đ...",https://vneconomy.vn/hang-ve-tranh-ban-thi-tru...,[],False,0.070348,"1/3,cuối,2,hôm,Khoảng,Kết phiên,15,87 điểm,T,s..."


## Cell 3 — Apply "passage: " Prefix

> **Required by intfloat/multilingual-e5-large**  
> Documents must be prefixed with `"passage: "` and queries with `"query: "`.  
> Missing this prefix significantly degrades retrieval quality.

In [4]:
from src.embedding import PASSAGE_PREFIX

# Apply prefix to every chunk text
prefixed_texts = [PASSAGE_PREFIX + str(t) for t in df_chunks['text'].tolist()]

print(f"Total texts to embed : {len(prefixed_texts):,}")
print(f"Prefix applied       : '{PASSAGE_PREFIX}'")
print(f"Sample (first 80 chars): {prefixed_texts[0][:80]}...")

print("\nCell 3 complete.")

Total texts to embed : 45,764
Prefix applied       : 'passage: '
Sample (first 80 chars): passage: Title: Hàng về tranh bán, thị trường chìm trong sắc đỏ, VN-Index mất sạ...

Cell 3 complete.


## Cell 4 — Load Embedding Model

In [5]:
from sentence_transformers import SentenceTransformer

model_name = emb_cfg['model_name']
print(f"Loading model: {model_name}")
print("(First run downloads ~2.2 GB — subsequent runs load from cache)")

embedding_model = SentenceTransformer(model_name, device=device)
embedding_model.max_seq_length = 512

print(f"\n✅ Model loaded on {device}")
print(f"   Embedding dimension : {embedding_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length : {embedding_model.max_seq_length}")

Loading model: intfloat/multilingual-e5-large
(First run downloads ~2.2 GB — subsequent runs load from cache)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]


✅ Model loaded on cuda
   Embedding dimension : 1024
   Max sequence length : 512


/tmp/ipykernel_2787/414819255.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Embedding dimension : {embedding_model.get_sentence_embedding_dimension()}")


## Cell 5 — Encode Chunks (with Checkpointing)

> ⏱️ **TIME WARNING**: ~2–3 hours on T4 for fixed_size (~42K chunks).  
> If the session times out, just re-run this cell — it resumes from the checkpoint automatically.

In [6]:
from src.embedding import encode_chunks_with_checkpoint

batch_size       = emb_cfg.get('batch_size', 64)
checkpoint_every = emb_cfg.get('checkpoint_every', 100)

print(f"Batch size      : {batch_size}")
print(f"Checkpoint every: {checkpoint_every} batches")
print(f"Total chunks    : {len(prefixed_texts):,}")
print(f"Total batches   : {(len(prefixed_texts) + batch_size - 1) // batch_size:,}")

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        ckpt = json.load(f)
    print(f"\n🔄 Resuming — {ckpt.get('completed_chunks', 0)}/{len(prefixed_texts)} chunks done")
else:
    print("\n🚀 Starting fresh encoding run...")

embeddings_mmap = encode_chunks_with_checkpoint(
    texts=prefixed_texts,
    model=embedding_model,
    output_npy_path=EMB_NPY_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=batch_size,
    checkpoint_every=checkpoint_every,
)

print(f"\n✅ Encoding complete. Embedding matrix shape: {embeddings_mmap.shape}")

[2026-05-08 01:57:13] [INFO] src.embedding: Embedding matrix: /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy  shape=(45764, 1024)  mode=w+
INFO:src.embedding:Embedding matrix: /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy  shape=(45764, 1024)  mode=w+


Batch size      : 64
Checkpoint every: 100 batches
Total chunks    : 45,764
Total batches   : 716

🚀 Starting fresh encoding run...


Encoding chunks:  14%|█▍        | 99/716 [05:42<34:55,  3.40s/batch][2026-05-08 02:03:00] [INFO] src.embedding: Checkpoint saved — batch 99 (6400/45764 chunks)
INFO:src.embedding:Checkpoint saved — batch 99 (6400/45764 chunks)
Encoding chunks:  28%|██▊       | 199/716 [11:30<30:51,  3.58s/batch][2026-05-08 02:08:47] [INFO] src.embedding: Checkpoint saved — batch 199 (12800/45764 chunks)
INFO:src.embedding:Checkpoint saved — batch 199 (12800/45764 chunks)
Encoding chunks:  42%|████▏     | 299/716 [17:16<24:05,  3.47s/batch][2026-05-08 02:14:33] [INFO] src.embedding: Checkpoint saved — batch 299 (19200/45764 chunks)
INFO:src.embedding:Checkpoint saved — batch 299 (19200/45764 chunks)
Encoding chunks:  56%|█████▌    | 399/716 [23:02<17:55,  3.39s/batch][2026-05-08 02:20:19] [INFO] src.embedding: Checkpoint saved — batch 399 (25600/45764 chunks)
INFO:src.embedding:Checkpoint saved — batch 399 (25600/45764 chunks)
Encoding chunks:  70%|██████▉   | 499/716 [28:47<12:37,  3.49s/batch][2026-05


✅ Encoding complete. Embedding matrix shape: (45764, 1024)


## Cell 6 — Build & Save FAISS Index

In [3]:
from src.embedding import build_faiss_index

if os.path.exists(FAISS_PATH):
    print(f"FAISS index already exists: {FAISS_PATH}")
    print("Loading existing index...")
    import faiss
    index = faiss.read_index(FAISS_PATH)
    print(f"✅ Loaded — {index.ntotal:,} vectors")
else:
    print("Building FAISS index...")
    index = build_faiss_index(
        npy_path=EMB_NPY_PATH,
        index_output_path=FAISS_PATH,
    )
    print(f"\n✅ FAISS index built and saved — {index.ntotal:,} vectors")

print(f"Index type : {type(index).__name__}")
print(f"Dimension  : {index.d}")

[2026-05-08 02:46:45] [INFO] src.embedding: Loading raw embeddings from: /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy
INFO:src.embedding:Loading raw embeddings from: /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy
[2026-05-08 02:46:45] [INFO] src.embedding: Embedding matrix shape inferred: (45764, 1024)
INFO:src.embedding:Embedding matrix shape inferred: (45764, 1024)


Building FAISS index...


[2026-05-08 02:46:45] [INFO] src.embedding: Normalizing vectors for Inner Product (Cosine Similarity simulation)...
INFO:src.embedding:Normalizing vectors for Inner Product (Cosine Similarity simulation)...
[2026-05-08 02:46:45] [INFO] src.embedding: Building FAISS IndexFlatIP with dimension 1024...
INFO:src.embedding:Building FAISS IndexFlatIP with dimension 1024...
[2026-05-08 02:46:45] [INFO] src.embedding: Saving FAISS index to /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/index.faiss
INFO:src.embedding:Saving FAISS index to /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/index.faiss



✅ FAISS index built and saved — 45,764 vectors
Index type : IndexFlatIP
Dimension  : 1024


## Cell 7 — Align & Save Metadata

In [5]:
import os
import pandas as pd
from src.embedding import align_and_save_metadata

# Nạp lại df_chunks từ ổ cứng nếu biến chưa tồn tại trong RAM
if 'df_chunks' not in locals():
    print(f"Loading chunks metadata from: {CHUNKS_PATH}")
    df_chunks = pd.read_parquet(CHUNKS_PATH)

chunk_ids_path = os.path.join(IDX_DIR, 'chunk_ids.json')
metadata_path = os.path.join(IDX_DIR, 'metadata.parquet')

if os.path.exists(chunk_ids_path) and os.path.exists(metadata_path):
    print("Metadata already aligned and saved. Skipping.")
else:
    print("Aligning and saving metadata...")
    ids_path, meta_path = align_and_save_metadata(
        df_chunks=df_chunks,
        output_dir=IDX_DIR,
    )
    print(f"✅ chunk_ids saved to: {ids_path}")
    print(f"✅ metadata saved to: {meta_path}")

# Verify alignment
import json
import faiss

idx = faiss.read_index(FAISS_PATH)
with open(chunk_ids_path, 'r') as f:
    chunk_ids = json.load(f)
meta_df = pd.read_parquet(metadata_path)

assert idx.ntotal == len(chunk_ids) == len(meta_df), "❌ LỖI: Số lượng record giữa FAISS, JSON và Parquet không khớp!"
print(f"✅ Alignment verified 3-ways: {idx.ntotal} records.")

Loading chunks metadata from: /content/drive/MyDrive/rag-vn-finance/data/chunks/fixed_size/chunks.parquet


[2026-05-08 02:48:48] [INFO] src.embedding: chunk_ids.json saved: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/chunk_ids.json  (45764 IDs)
INFO:src.embedding:chunk_ids.json saved: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/chunk_ids.json  (45764 IDs)


Aligning and saving metadata...


[2026-05-08 02:48:48] [INFO] src.embedding: metadata.parquet saved: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/metadata.parquet  (45764 rows)
INFO:src.embedding:metadata.parquet saved: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/metadata.parquet  (45764 rows)
[2026-05-08 02:48:48] [INFO] src.embedding: Row-alignment verified: chunk_ids ↔ metadata ↔ FAISS index.
INFO:src.embedding:Row-alignment verified: chunk_ids ↔ metadata ↔ FAISS index.


✅ chunk_ids saved to: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/chunk_ids.json
✅ metadata saved to: /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/metadata.parquet
✅ Alignment verified 3-ways: 45764 records.


## Cell 8 — Smoke Test: Query the Index

In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Reload model & index (handles session resets cleanly)
if 'embedding_model' not in dir() or embedding_model is None:
    embedding_model = SentenceTransformer(emb_cfg['model_name'], device=device)

idx      = faiss.read_index(FAISS_PATH)
df_meta  = pd.read_parquet(meta_path)

# Test query with the mandatory "query: " prefix
TEST_QUERY = "query: Lãi suất ngân hàng Việt Nam năm 2023"

query_emb = embedding_model.encode(TEST_QUERY, normalize_embeddings=True).reshape(1, -1).astype('float32')
scores, faiss_ids = idx.search(query_emb, 5)

print(f"Query: {TEST_QUERY}")
print(f"\nTop-5 results:")
for rank, (fid, score) in enumerate(zip(faiss_ids[0], scores[0]), 1):
    row = df_meta.iloc[fid]
    print(f"  #{rank}  score={score:.4f}  chunk_id={row['chunk_id']}")
    print(f"       title  : {row['title'][:70]}...")
    print(f"       year   : {row['year']}  source: {row['source']}")

print("\n✅ Smoke test passed.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: query: Lãi suất ngân hàng Việt Nam năm 2023

Top-5 results:
  #1  score=0.8921  chunk_id=6fbef3d751c05a88_c0004
       title  : Cổ phiếu ngân hàng đang hẫp dẫn để đầu tư nhưng tới 2023 rủi ro có thể...
       year   : 2022  source: vneconomy.vn
  #2  score=0.8863  chunk_id=ec73d491a3ae042d_c0001
       title  : Lãi suất cho vay mới đã giảm 0,5% đến 3% so với cuối năm 2022...
       year   : 2023  source: vneconomy.vn
  #3  score=0.8841  chunk_id=6bc30e5296673e5c_c0003
       title  : Ngành ngân hàng tiếp tục đối mặt loạt khó khăn trong năm 2023...
       year   : 2023  source: vneconomy.vn
  #4  score=0.8836  chunk_id=edfbbc447d48af95_c0000
       title  : Năm 2022 và 2023 sẽ giảm 1% lãi vay ngân hàng?...
       year   : 2022  source: vneconomy.vn
  #5  score=0.8834  chunk_id=6fbef3d751c05a88_c0003
       title  : Cổ phiếu ngân hàng đang hẫp dẫn để đầu tư nhưng tới 2023 rủi ro có thể...
       year   : 2022  source: vneconomy.vn

✅ Smoke test passed.


## Cell 9 — Cleanup Temporary .npy File

> **Run this cell ONLY after verifying the FAISS index is correct in Cell 8.**  
> The intermediate `.npy` embedding file is no longer needed once the FAISS index is built.  
> Deleting it frees ~200–400 MB of Drive space per strategy.

In [7]:
# Safety guard: only delete if FAISS index exists and is non-empty
import faiss

idx_check = faiss.read_index(FAISS_PATH)
assert idx_check.ntotal > 0, "FAISS index is empty — do NOT delete the .npy file!"

if os.path.exists(EMB_NPY_PATH):
    npy_size_mb = os.path.getsize(EMB_NPY_PATH) / 1e6
    os.remove(EMB_NPY_PATH)
    print(f"✅ Deleted: {EMB_NPY_PATH}  ({npy_size_mb:.0f} MB freed)")
else:
    print(f"ℹ️  File already removed: {EMB_NPY_PATH}")

print(f"\nFinal outputs for strategy '{TARGET_STRATEGY}':")
for f in [FAISS_PATH, meta_path, ids_path, CHECKPOINT_PATH]:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f"  {'✅' if os.path.exists(f) else '❌'}  {f}  ({size:.1f} MB)")

print(f"\n✅ Phase 3 complete for strategy: {TARGET_STRATEGY}")
print("Confirm 'Xong' before proceeding to Phase 4 (BM25 Indexing).")

✅ Deleted: /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/embeddings.npy  (187 MB freed)

Final outputs for strategy 'fixed_size':
  ✅  /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/index.faiss  (187.4 MB)
  ✅  /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/metadata.parquet  (27.8 MB)
  ✅  /content/drive/MyDrive/rag-vn-finance/indexes/fixed_size/chunk_ids.json  (1.2 MB)
  ✅  /content/drive/MyDrive/rag-vn-finance/embeddings/fixed_size/checkpoint.json  (0.0 MB)

✅ Phase 3 complete for strategy: fixed_size
Confirm 'Xong' before proceeding to Phase 4 (BM25 Indexing).
